# NB14 — Stacking Ensemble Deneyi (v2)

**TEKNOFEST Sağlıkta Yapay Zeka** | Genetik Varyant Patojenite Tahmini

## Değişiklik özeti (v2)
* **A1** Meta probability üzerinde `optimize_threshold()` (F1-max grid).
* **A2** NB12 XGBoost+OHE pipeline'ı port edildi; F1≈0.9020 baseline doğrulanır.
* **A3** Sabit sütunlar + birebir özdeş sütun çiftleri (`CAT_3`/`CAT_5` dahil) train sonrası drop.
* **B1** 7 baz model → 5 çeşitlilik (CatBoost, LightGBM, L1-LR, TabNet, kNN).
* **B3** OOF probability'lere isotonic calibration; meta-feature matrisine kalibre edilmiş prob'lar girer.
* **B2 split** düzeltildi: B1 ile aynı test set, train'de pathogenic undersample, fazla pathogenic **DROP** edilir (test'e gitmez → leakage yok).

## Deney Matrisi (2 × 5 = 10 Konfigurasyon)
| Balance | Açıklama |
|---|---|
| B1 | Stratified 80/20 |
| B2 | B1 test set sabit; train'de patho. undersample, fazla satır atılır |

Missing strategy: M1–M5 (medyan / drop / flag+medyan / flag+drop / hibrit).


In [1]:
# Cell 1: Imports & Setup
# IMPORTANT: libomp double-load guard + thread caps (PyTorch/LightGBM)
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '4')
os.environ.setdefault('MKL_NUM_THREADS', '4')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '4')
os.environ.setdefault('VECLIB_MAXIMUM_THREADS', '4')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '4')

import sys, os, warnings, json, time
from copy import deepcopy
from datetime import datetime
from itertools import product as itertools_product

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score
import torch
from fpdf import FPDF

# Path setup - works whether running from project root or notebooks/
_cwd = os.getcwd()
_root = _cwd if os.path.basename(_cwd) == 'teknofest_model' else os.path.dirname(_cwd)
if _root not in sys.path:
    sys.path.insert(0, _root)

from config import (SEED, TEST_SIZE, PROJECT_ROOT, REPORTS_DIR,
                    RESULTS_STACKING_DIR, MODELS_STACKING_DIR)
from src.columns_real import (
    AL_COLS, CAT_COLS, EK_COLS, AA_COLS, ALL_FEATURE_COLS,
    NON_FEATURE_COLS, PANEL_INFO, get_constant_cols, get_duplicate_col_pairs,
)
from src.metrics import compute_all_metrics, optimize_threshold
from src.models import (
    MLP3Layer, DeepMLP,
    NN_FIXED_FAST, DNN_FIXED_FAST,
    grid_search_nn_fast, grid_search_dnn_fast,
)
# (XGBoost import KALDIRILDI -- A2 subprocess olarak calistirilir)

# Constants
N_OOF_FOLDS = 5
BALANCE_IDS = ['B1', 'B2']
MISSING_IDS = ['M1', 'M3', 'M5']  # M2/M4 dropped: MASTER'da NaN-suz sütun yok
NUM_FEAT_COLS = set(AL_COLS + EK_COLS)
CAT_FEAT_COLS = set(CAT_COLS + AA_COLS)
AA_UNKNOWN   = 'X'

os.makedirs(RESULTS_STACKING_DIR, exist_ok=True)
os.makedirs(MODELS_STACKING_DIR,  exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'SEED={SEED}, TEST_SIZE={TEST_SIZE}, OOF folds={N_OOF_FOLDS}')


Project root : /Users/tefe/teknofest_model/teknofest_model
SEED=42, TEST_SIZE=0.2, OOF folds=5


In [2]:
# Cell 2: Data Loading
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'real_data')
ID_COL, LABEL_COL = 'Variant_ID', 'Label'

def load_panel(fname):
    path = os.path.join(DATA_DIR, fname)
    df = pd.read_csv(path)
    feat_cols = [c for c in ALL_FEATURE_COLS if c in df.columns]
    X = df[feat_cols].copy()
    y = df[LABEL_COL].copy()
    ids = df[ID_COL].copy() if ID_COL in df.columns else pd.Series(range(len(df)))
    return X, y, ids

X_master, y_master, id_master = load_panel('YARISMA_TRAIN_MASTER.csv')
X_cftr,   y_cftr,   _         = load_panel('YARISMA_TRAIN_CFTR.csv')
X_kanser, y_kanser, _         = load_panel('YARISMA_TRAIN_KANSER.csv')
X_pah,    y_pah,    _         = load_panel('YARISMA_TRAIN_PAH.csv')

for name, X, y in [('MASTER', X_master, y_master), ('CFTR', X_cftr, y_cftr),
                    ('KANSER', X_kanser, y_kanser), ('PAH', X_pah, y_pah)]:
    pos, neg = int((y==1).sum()), int((y==0).sum())
    miss = X.isna().mean().mean()
    print(f'{name:8s}: n={len(y):4d}  pos={pos:4d} ({100*pos/len(y):.1f}%)  '
          f'neg={neg:4d}  avg_miss={100*miss:.1f}%  ncols={X.shape[1]}')


MASTER  : n=2931  pos=2149 (73.3%)  neg= 782  avg_miss=55.3%  ncols=351
CFTR    : n= 111  pos=  90 (81.1%)  neg=  21  avg_miss=30.5%  ncols=351
KANSER  : n= 388  pos= 268 (69.1%)  neg= 120  avg_miss=57.5%  ncols=351
PAH     : n= 372  pos= 310 (83.3%)  neg=  62  avg_miss=54.6%  ncols=351


In [3]:
# Cell 3: Balance Split Functions
# B1: standart stratified 80/20.
# B2 (FIXED): Once B1 split yap, sonra TRAIN icinde pathogenic'i undersample et;
# fazla pathogenic satirlari DROP edilir (test'e tasinmaz). Boylece B1 ile
# AYNI test seti uzerinde calismaya devam ediyoruz; train ID'leri test'te
# ASLA bulunmaz.

def make_b1_split(X, y):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
    )
    return X_tr, X_te, y_tr, y_te


def make_b2_split(X, y):
    # B1 ile ayni split (test set degismez)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
    )
    n_benign  = int((y_tr == 0).sum())
    patho_idx = y_tr[y_tr == 1].index.to_numpy()
    rng = np.random.default_rng(SEED)
    rng.shuffle(patho_idx)
    keep_patho = patho_idx[:n_benign]  # fazla pathogenic'i AT (test'e tasinmaz)

    keep_all = np.concatenate([y_tr[y_tr == 0].index.to_numpy(), keep_patho])
    X_tr_b2 = X_tr.loc[keep_all]
    y_tr_b2 = y_tr.loc[keep_all]

    # Sanity: train/test ID kesisimi sifir olmali
    assert len(set(X_tr_b2.index) & set(X_te.index)) == 0, 'B2 train/test overlap!'
    return X_tr_b2, X_te, y_tr_b2, y_te


for bal_id, fn in [('B1', make_b1_split), ('B2', make_b2_split)]:
    Xtr, Xte, ytr, yte = fn(X_master, y_master)
    overlap = len(set(Xtr.index) & set(Xte.index))
    print(f'{bal_id}: train n={len(ytr):4d} pos={int((ytr==1).sum()):4d} '
          f'neg={int((ytr==0).sum()):4d}  | test n={len(yte):4d} '
          f'pos={int((yte==1).sum()):4d} neg={int((yte==0).sum()):4d}  '
          f'| ID overlap={overlap}')


B1: train n=2344 pos=1719 neg= 625  | test n= 587 pos= 430 neg= 157  | ID overlap=0
B2: train n=1250 pos= 625 neg= 625  | test n= 587 pos= 430 neg= 157  | ID overlap=0


In [4]:
# Cell 4: Preprocessing Functions (5 Missing Scenarios)
def fit_preprocessor(X_train, scenario):
    nan_mask = X_train.isna()
    nan_cols = [c for c in X_train.columns if nan_mask[c].any()]
    nan_frac = {c: float(nan_mask[c].mean()) for c in nan_cols}
    num_nan  = [c for c in nan_cols if c in NUM_FEAT_COLS]

    prep = {'scenario': scenario, 'nan_cols': nan_cols, 'nan_frac': nan_frac,
            'drop_cols': [], 'flag_cols': [], 'medians': {}}

    if scenario == 'M1':
        prep['medians'] = {c: float(X_train[c].median()) for c in num_nan}
    elif scenario == 'M3':
        prep['flag_cols'] = list(nan_cols)
        prep['medians']   = {c: float(X_train[c].median()) for c in num_nan}
    elif scenario == 'M5':
        high_nan_num = [c for c in num_nan if nan_frac[c] > 0.5]
        low_nan_num  = [c for c in num_nan if nan_frac[c] <= 0.5]
        prep['flag_cols'] = list(nan_cols)
        prep['drop_cols'] = list(high_nan_num)
        prep['medians']   = {c: float(X_train[c].median()) for c in low_nan_num}
    return prep


def apply_preprocessor(X, prep):
    X = X.copy()
    for col in prep['flag_cols']:
        if col in X.columns:
            X[f'is_missing_{col}'] = X[col].isna().astype(np.float32)
    for col, med in prep['medians'].items():
        if col in X.columns:
            X[col] = X[col].fillna(med)
    to_drop = [c for c in prep['drop_cols'] if c in X.columns]
    if to_drop:
        X = X.drop(columns=to_drop)
    for col in list(X.columns):
        if col in CAT_FEAT_COLS and col in X.columns:
            fill_val = AA_UNKNOWN if col in set(AA_COLS) else 'MISSING'
            X[col] = X[col].fillna(fill_val).astype(str)
    num_cols_x = X.select_dtypes(include=[np.number]).columns
    X[num_cols_x] = X[num_cols_x].fillna(0)
    return X


def get_surviving_cat_cols(prep):
    dropped = set(prep['drop_cols'])
    return [c for c in list(CAT_FEAT_COLS) if c not in dropped]


# A3: train sonrasi sabit ve birebir ayni sutun ciftlerini tespit et,
# ikinci adi DROP listesine ekle. Cagri yeri: per-config preprocessing sonrasi.
def detect_redundant_cols(X_train):
    consts = get_constant_cols(X_train)
    dups   = get_duplicate_col_pairs(X_train)
    dup_drop = [b for _, b in dups]  # ikincisini at
    return list(set(consts) | set(dup_drop)), consts, dups


print('Preprocessor sanity check:')
Xtr_tmp, Xte_tmp, ytr_tmp, yte_tmp = make_b1_split(X_master, y_master)
for sc in MISSING_IDS:
    p = fit_preprocessor(Xtr_tmp, sc)
    Xtr_p = apply_preprocessor(Xtr_tmp, p)
    redund, consts, dups = detect_redundant_cols(Xtr_p)
    cats = get_surviving_cat_cols(p)
    flag_count = len([c for c in Xtr_p.columns if c.startswith('is_missing_')])
    print(f'  {sc}: feat={Xtr_p.shape[1]:4d} cat={len(cats):2d} flag={flag_count:3d}'
          f' drop={len(p["drop_cols"]):3d} consts={len(consts):2d} dup_pairs={len(dups):2d}')


Preprocessor sanity check:
  M1: feat= 351 cat= 8 flag=  0 drop=  0 consts=57 dup_pairs=1557
  M3: feat= 702 cat= 8 flag=351 drop=  0 consts=57 dup_pairs=11447
  M5: feat= 539 cat= 8 flag=351 drop=163 consts=37 dup_pairs=10557


In [5]:
# Cell 5 (A2): NB12 XGBoost+OHE baseline -- isolated subprocess
# macOS arm64'te xgboost'un wheel-icinde-gomulu libomp.dylib'i notebook
# surecindeki lightgbm/torch libomp ile ayni adres uzayinda cakisip SIGSEGV
# uretir. Bu yuzden A2'yi subprocess'te calistirip JSON ile sonuc okuyoruz.
# Notebook surecine XGBoost ASLA import edilmez.

import subprocess as _sp
import json as _json

_A2_SCRIPT = os.path.join(PROJECT_ROOT, 'scripts', 'a2_nb12_baseline.py')
print(f'=== A2: NB12 XGBoost+OHE baseline (isolated subprocess) ===')
print(f'  runner: {_A2_SCRIPT}')

_t0 = time.time()
_proc = _sp.run(
    [sys.executable, _A2_SCRIPT, PROJECT_ROOT],
    capture_output=True, text=True,
)
_elapsed = time.time() - _t0
print(f'  subprocess elapsed: {_elapsed:.1f}s (rc={_proc.returncode})')

if _proc.stderr:
    # Forward subprocess stderr (preprocessing shape info, grid search progress)
    for _line in _proc.stderr.strip().splitlines()[-20:]:
        print(f'  [sub] {_line}')

NB12_BASELINE = None
if _proc.returncode == 0 and '==A2_RESULT_JSON==' in _proc.stdout:
    _payload = _proc.stdout.split('==A2_RESULT_JSON==', 1)[1].strip().splitlines()[0]
    NB12_BASELINE = _json.loads(_payload)
    print(f'\n  XGBoost+OHE thr={NB12_BASELINE["thr"]:.3f}  '
          f'F1={NB12_BASELINE["f1"]:.4f}  AUC={NB12_BASELINE["auc"]:.4f}  '
          f'P={NB12_BASELINE["precision"]:.4f}  R={NB12_BASELINE["recall"]:.4f}')
    print(f'  NB12 hedef: F1=0.9020 AUC=0.8404')
else:
    print(f'  A2 subprocess BASARISIZ -- NB12_BASELINE None olarak isaretlendi.')
    print(f'  stdout tail:')
    for _line in _proc.stdout.strip().splitlines()[-10:]:
        print(f'    {_line}')
    NB12_BASELINE = {'f1': float("nan"), 'auc': float("nan"),
                     'thr': 0.5, 'error': 'subprocess_failed'}


=== A2: NB12 XGBoost+OHE baseline (isolated subprocess) ===
  runner: /Users/tefe/teknofest_model/teknofest_model/scripts/a2_nb12_baseline.py
  subprocess elapsed: 17.3s (rc=0)
  [sub] NB12 OHE matrix: train=(2344, 618) test=(587, 618)

  XGBoost+OHE thr=0.310  F1=0.8877  AUC=0.8388  P=0.8431  R=0.9372
  NB12 hedef: F1=0.9020 AUC=0.8404


In [6]:
# Cell 6: 6-Model Base Set (B1 diversity) -- OOF helpers + wrappers
# Modeller:
#   1) CatBoost          (native categorical, is_missing_* OLMADAN)
#   2) LightGBM (LE)     (full feature, is_missing_* DAHIL)
#   3) Logistic L1       (OHE + StandardScaler)
#   4) MLP3Layer (NN)    (LE + scaler, 3-katmanli MLP)
#   5) DeepMLP (DNN)     (LE + scaler, derin MLP, dropout-li)
#   6) k-NN              (OHE + StandardScaler; non-parametric)
# NOT: TabNet macOS arm64'te OMP_NUM_THREADS=1 ile pratik degildi
# (10 konfig x 6 fit -> 750 dk asti); MLP3/DeepMLP ile degistirildi.
BASE_MODEL_ORDER = ['catboost', 'lightgbm', 'logreg_l1', 'mlp3', 'deep_mlp', 'knn']

# ---------- LightGBM params ----------
LGBM_FIXED_LOCAL = {
    'objective': 'binary', 'verbosity': -1,
    'random_state': SEED, 'class_weight': 'balanced',
    'n_estimators': 300, 'num_leaves': 63, 'learning_rate': 0.05,
}
LGBM_GRID_LOCAL = {
    'num_leaves':     [31, 63],
    'learning_rate':  [0.05, 0.1],
    'n_estimators':   [200, 400],
}

# ---------- CatBoost params ----------
CB_FIXED_LOCAL = {
    'iterations': 500, 'depth': 6, 'learning_rate': 0.05,
    'loss_function': 'Logloss', 'eval_metric': 'F1',
    'random_seed': SEED, 'auto_class_weights': 'Balanced',
    'verbose': False,
}
CB_GRID_LOCAL = {
    'depth':         [4, 6, 8],
    'learning_rate': [0.05, 0.1],
}

# ---------- Logistic L1 params ----------
LR_GRID = {'C': [0.05, 0.1, 0.5, 1.0, 5.0]}


# ---------- kNN params ----------
KNN_GRID = {'n_neighbors': [5, 11, 21, 41]}


# ---------- encoding helpers ----------
def _le_encode(X, cat_cols):
    X = X.copy()
    encoders = {}
    for col in cat_cols:
        if col in X.columns:
            le = LabelEncoder()
            le.fit(X[col].astype(str))
            known = set(le.classes_)
            X[col] = X[col].astype(str).map(
                lambda v, le=le, k=known: le.transform([v])[0] if v in k else -1
            )
            encoders[col] = le
    return X, encoders


def _le_transform(X, encoders):
    X = X.copy()
    for col, le in encoders.items():
        if col in X.columns:
            known = set(le.classes_)
            X[col] = X[col].astype(str).map(
                lambda v, le=le, k=known: le.transform([v])[0] if v in k else -1
            )
    return X


def _prep_catboost(X, cat_cols):
    X = X.copy()
    for col in X.columns:
        if col in set(cat_cols):
            X[col] = X[col].astype(str)
        else:
            X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0).astype(float)
    return X


def _ohe_encode(X_tr, X_te, cat_cols):
    cat_present = [c for c in cat_cols if c in X_tr.columns]
    num_cols    = [c for c in X_tr.columns if c not in set(cat_present)]
    X_tr_num = X_tr[num_cols].values.astype(np.float32)
    X_te_num = X_te[num_cols].values.astype(np.float32)
    if cat_present:
        ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore', dtype=np.float32)
        X_tr_cat = ohe.fit_transform(X_tr[cat_present].astype(str))
        X_te_cat = ohe.transform(X_te[cat_present].astype(str))
        X_tr_arr = np.hstack([X_tr_num, X_tr_cat])
        X_te_arr = np.hstack([X_te_num, X_te_cat])
    else:
        ohe = None
        X_tr_arr, X_te_arr = X_tr_num, X_te_num
    return X_tr_arr, X_te_arr, ohe, cat_present, num_cols


# ---------- CV / grid search for each base ----------
def _cv_f1(model_factory, X_tr_arr, y_tr, n_splits=3):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    f1s = []
    for tr_idx, val_idx in skf.split(X_tr_arr, y_tr):
        m = model_factory()
        m.fit(X_tr_arr[tr_idx], y_tr.iloc[tr_idx])
        prob = m.predict_proba(X_tr_arr[val_idx])[:, 1]
        f1s.append(f1_score(y_tr.iloc[val_idx], (prob >= 0.5).astype(int),
                            zero_division=0))
    return float(np.mean(f1s))


def _oof_predict_generic(X_tr_arr, y_tr, fit_predict_fold):
    skf = StratifiedKFold(n_splits=N_OOF_FOLDS, shuffle=True, random_state=SEED)
    oof = np.zeros(len(y_tr))
    for tr_idx, val_idx in skf.split(X_tr_arr, y_tr):
        oof[val_idx] = fit_predict_fold(
            X_tr_arr[tr_idx], y_tr.iloc[tr_idx], X_tr_arr[val_idx]
        )
    return oof


def train_catboost(X_train, y_train, X_test, y_test, cat_cols):
    X_tr_cb = _prep_catboost(X_train, cat_cols)
    X_te_cb = _prep_catboost(X_test,  cat_cols)
    cat_idx = [list(X_tr_cb.columns).index(c) for c in cat_cols if c in X_tr_cb.columns]
    best_f1, best_combo = -1, None
    for d in CB_GRID_LOCAL['depth']:
        for lr in CB_GRID_LOCAL['learning_rate']:
            params = {**CB_FIXED_LOCAL, 'depth': d, 'learning_rate': lr}
            skf = StratifiedKFold(3, shuffle=True, random_state=SEED)
            fs = []
            for tr_idx, val_idx in skf.split(X_tr_cb, y_train):
                m = CatBoostClassifier(**params)
                m.fit(X_tr_cb.iloc[tr_idx], y_train.iloc[tr_idx],
                      cat_features=cat_idx, silent=True)
                p = m.predict_proba(X_tr_cb.iloc[val_idx])[:, 1]
                fs.append(f1_score(y_train.iloc[val_idx], (p >= 0.5).astype(int),
                                    zero_division=0))
            mf = float(np.mean(fs))
            if mf > best_f1:
                best_f1, best_combo = mf, {'depth': d, 'learning_rate': lr}
    print(f'    CatBoost best={best_combo} cv_f1={best_f1:.4f}')

    final = CatBoostClassifier(**{**CB_FIXED_LOCAL, **best_combo})
    final.fit(X_tr_cb, y_train, cat_features=cat_idx, silent=True)
    test_proba = final.predict_proba(X_te_cb)[:, 1]

    # OOF
    skf = StratifiedKFold(N_OOF_FOLDS, shuffle=True, random_state=SEED)
    oof = np.zeros(len(y_train))
    for tr_idx, val_idx in skf.split(X_tr_cb, y_train):
        m = CatBoostClassifier(**{**CB_FIXED_LOCAL, **best_combo})
        m.fit(X_tr_cb.iloc[tr_idx], y_train.iloc[tr_idx],
              cat_features=cat_idx, silent=True)
        oof[val_idx] = m.predict_proba(X_tr_cb.iloc[val_idx])[:, 1]

    def predict_fn(X, m=final, cc=cat_cols):
        return m.predict_proba(_prep_catboost(X, cc))[:, 1]
    return final, test_proba, oof, predict_fn


def train_lightgbm(X_train, y_train, X_test, y_test, cat_cols):
    X_tr_enc, enc = _le_encode(X_train, cat_cols)
    X_te_enc      = _le_transform(X_test, enc)
    arr_tr = X_tr_enc.values.astype(np.float32)
    arr_te = X_te_enc.values.astype(np.float32)

    best_f1, best_combo = -1, None
    for nl in LGBM_GRID_LOCAL['num_leaves']:
        for lr in LGBM_GRID_LOCAL['learning_rate']:
            for ne in LGBM_GRID_LOCAL['n_estimators']:
                params = {**LGBM_FIXED_LOCAL,
                          'num_leaves': nl, 'learning_rate': lr, 'n_estimators': ne}
                mf = _cv_f1(lambda p=params: lgb.LGBMClassifier(**p), arr_tr, y_train)
                if mf > best_f1:
                    best_f1, best_combo = mf, {'num_leaves': nl,
                                                'learning_rate': lr, 'n_estimators': ne}
    print(f'    LightGBM best={best_combo} cv_f1={best_f1:.4f}')

    final = lgb.LGBMClassifier(**{**LGBM_FIXED_LOCAL, **best_combo})
    final.fit(arr_tr, y_train)
    test_proba = final.predict_proba(arr_te)[:, 1]

    def fp_fold(Xtr, ytr, Xva):
        m = lgb.LGBMClassifier(**{**LGBM_FIXED_LOCAL, **best_combo})
        m.fit(Xtr, ytr)
        return m.predict_proba(Xva)[:, 1]
    oof = _oof_predict_generic(arr_tr, y_train, fp_fold)

    def predict_fn(X, m=final, enc=enc):
        Xe = _le_transform(X, enc).values.astype(np.float32)
        return m.predict_proba(Xe)[:, 1]
    return final, test_proba, oof, predict_fn


def train_logreg_l1(X_train, y_train, X_test, y_test, cat_cols):
    X_tr_arr, X_te_arr, ohe, cat_present, num_cols = _ohe_encode(X_train, X_test, cat_cols)
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr_arr)
    X_te_s = sc.transform(X_te_arr)

    best_f1, best_C = -1, 1.0
    for C in LR_GRID['C']:
        mf = _cv_f1(
            lambda C=C: LogisticRegression(penalty='l1', solver='liblinear',
                                            C=C, class_weight='balanced',
                                            max_iter=2000, random_state=SEED),
            X_tr_s, y_train,
        )
        if mf > best_f1:
            best_f1, best_C = mf, C
    print(f'    LR-L1 best C={best_C} cv_f1={best_f1:.4f}')

    final = LogisticRegression(penalty='l1', solver='liblinear', C=best_C,
                                class_weight='balanced', max_iter=2000,
                                random_state=SEED)
    final.fit(X_tr_s, y_train)
    test_proba = final.predict_proba(X_te_s)[:, 1]

    def fp_fold(Xtr, ytr, Xva):
        m = LogisticRegression(penalty='l1', solver='liblinear', C=best_C,
                                class_weight='balanced', max_iter=2000,
                                random_state=SEED)
        m.fit(Xtr, ytr)
        return m.predict_proba(Xva)[:, 1]
    oof = _oof_predict_generic(X_tr_s, y_train, fp_fold)

    def predict_fn(X, m=final, ohe=ohe, cp=cat_present, nc=num_cols, sc=sc):
        Xn = X[nc].values.astype(np.float32) if nc else np.empty((len(X), 0), dtype=np.float32)
        if ohe is not None and cp:
            Xc = ohe.transform(X[cp].astype(str))
            Xa = np.hstack([Xn, Xc])
        else:
            Xa = Xn
        return m.predict_proba(sc.transform(Xa))[:, 1]
    return final, test_proba, oof, predict_fn


# NN/DNN -- subprocess runner (libomp deadlock fix)
# Main kernel libomp'unu yormadan NN/DNN fit'i izole bir Python sureci icinde
# yapilir. Bkz scripts/nn_dnn_runner.py.
import pickle as _pickle
import subprocess as _sp

_NN_RUNNER_PATH = os.path.join(PROJECT_ROOT, 'scripts', 'nn_dnn_runner.py')


def _call_nn_runner(payload):
    proc = _sp.run(
        [sys.executable, _NN_RUNNER_PATH, PROJECT_ROOT],
        input=_pickle.dumps(payload),
        capture_output=True,
    )
    if proc.returncode != 0:
        print('[nn-runner] FAILED. stderr tail:')
        for ln in proc.stderr.decode(errors='replace').splitlines()[-15:]:
            print(' ', ln)
        raise RuntimeError(f'nn_dnn_runner failed rc={proc.returncode}')
    # Forward runner stderr (progress) tail
    for ln in proc.stderr.decode(errors='replace').splitlines()[-12:]:
        print(' ', ln)
    return _pickle.loads(proc.stdout)


def _build_nn_subproc_predict_fn(model_type, model_state_pkl, enc_pkl,
                                  scaler_pkl, input_dim, best_combo):
    """Panel predict -- yine subprocess'i tetikler."""
    def predict_fn(X_panel):
        payload = {
            'mode':            'predict',
            'model_type':      model_type,
            'X_panel':         X_panel,
            'model_state_pkl': model_state_pkl,
            'enc_pkl':         enc_pkl,
            'scaler_pkl':      scaler_pkl,
            'input_dim':       input_dim,
            'best_combo':      best_combo,
        }
        res = _call_nn_runner(payload)
        return res['panel_proba']
    return predict_fn


def _train_nn_subproc(model_type, X_train, y_train, X_test, y_test, cat_cols):
    payload = {
        'mode':       'fit',
        'model_type': model_type,
        'X_train':    X_train, 'y_train': y_train,
        'X_test':     X_test,  'y_test':  y_test,
        'cat_cols':   cat_cols,
    }
    res = _call_nn_runner(payload)
    predict_fn = _build_nn_subproc_predict_fn(
        model_type, res['model_state_pkl'], res['enc_pkl'], res['scaler_pkl'],
        res['input_dim'], res['best_combo'],
    )
    # `model` slot'unu sahte bir nesne ile dolduralim -- run_all_base_models
    # sonuclar dict'ine 'model' anahtari koyuyor, ama biz cikarmadik.
    fake_model = {'subprocess': True, 'type': model_type,
                   'best_combo': res['best_combo']}
    return fake_model, res['test_proba'], res['oof_proba'], predict_fn


def train_mlp3(X_train, y_train, X_test, y_test, cat_cols):
    return _train_nn_subproc('mlp3', X_train, y_train, X_test, y_test, cat_cols)


def train_deep_mlp(X_train, y_train, X_test, y_test, cat_cols):
    return _train_nn_subproc('deep_mlp', X_train, y_train, X_test, y_test, cat_cols)


def train_knn(X_train, y_train, X_test, y_test, cat_cols):
    X_tr_arr, X_te_arr, ohe, cat_present, num_cols = _ohe_encode(X_train, X_test, cat_cols)
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr_arr)
    X_te_s = sc.transform(X_te_arr)

    best_f1, best_k = -1, 11
    for k in KNN_GRID['n_neighbors']:
        mf = _cv_f1(lambda k=k: KNeighborsClassifier(n_neighbors=k, n_jobs=-1),
                    X_tr_s, y_train)
        if mf > best_f1:
            best_f1, best_k = mf, k
    print(f'    kNN best k={best_k} cv_f1={best_f1:.4f}')

    final = KNeighborsClassifier(n_neighbors=best_k, n_jobs=-1)
    final.fit(X_tr_s, y_train)
    test_proba = final.predict_proba(X_te_s)[:, 1]

    def fp_fold(Xtr, ytr, Xva):
        m = KNeighborsClassifier(n_neighbors=best_k, n_jobs=-1)
        m.fit(Xtr, ytr)
        return m.predict_proba(Xva)[:, 1]
    oof = _oof_predict_generic(X_tr_s, y_train, fp_fold)

    def predict_fn(X, m=final, ohe=ohe, cp=cat_present, nc=num_cols, sc=sc):
        Xn = X[nc].values.astype(np.float32) if nc else np.empty((len(X), 0), dtype=np.float32)
        if ohe is not None and cp:
            Xc = ohe.transform(X[cp].astype(str))
            Xa = np.hstack([Xn, Xc])
        else:
            Xa = Xn
        return m.predict_proba(sc.transform(Xa))[:, 1]
    return final, test_proba, oof, predict_fn


print('5-model base set defined.')


5-model base set defined.


In [7]:
# Cell 7: Run base models + Isotonic calibration (B3)
FIXED_THR = 0.5  # base model raw threshold for diagnostics only

BASE_TRAINERS = {
    'catboost':  lambda Xtr, ytr, Xte, yte, cc: train_catboost(
        # CatBoost is_missing_* OLMADAN: flag sutunlarini at
        Xtr.drop(columns=[c for c in Xtr.columns if c.startswith('is_missing_')]),
        ytr,
        Xte.drop(columns=[c for c in Xte.columns if c.startswith('is_missing_')]),
        yte, cc,
    ),
    'lightgbm':  train_lightgbm,
    'logreg_l1': train_logreg_l1,
    'mlp3':      train_mlp3,
    'deep_mlp':  train_deep_mlp,
    'knn':       train_knn,
}


def run_all_base_models(X_train, y_train, X_test, y_test, cat_cols_present):
    results = {}
    for i, mname in enumerate(BASE_MODEL_ORDER):
        print(f'[{i+1}/{len(BASE_MODEL_ORDER)}] {mname}')
        trainer = BASE_TRAINERS[mname]
        model, test_proba, oof_proba, predict_fn = trainer(
            X_train, y_train, X_test, y_test, cat_cols_present
        )

        # B3: Isotonic calibration -- fit on OOF (train) and apply to test probs
        iso = IsotonicRegression(out_of_bounds='clip', y_min=0.0, y_max=1.0)
        iso.fit(oof_proba, y_train.values)
        oof_calib  = iso.transform(oof_proba)
        test_calib = iso.transform(test_proba)

        # Metrics on calibrated probs (still 0.5 threshold for base diagnostics)
        met = compute_all_metrics(y_test.values, (test_calib >= FIXED_THR).astype(int), test_calib)
        results[mname] = {
            'model': model, 'predict_fn': predict_fn,
            'test_proba_raw':   test_proba,  'oof_proba_raw':   oof_proba,
            'test_proba_calib': test_calib,  'oof_proba_calib': oof_calib,
            'iso': iso,
            'metrics': met,
        }
        print(f'  F1={met["f1"]:.4f}  AUC={met["auc_roc"]:.4f}  '
              f'(raw_mean={test_proba.mean():.3f} calib_mean={test_calib.mean():.3f})')
    return results


print('run_all_base_models() defined. Base model order:', BASE_MODEL_ORDER)


run_all_base_models() defined. Base model order: ['catboost', 'lightgbm', 'logreg_l1', 'mlp3', 'deep_mlp', 'knn']


In [8]:
# Cell 8: Meta-Learner + Threshold Optimization (A1)
def build_meta_features(proba_dict, model_order=None):
    if model_order is None:
        model_order = BASE_MODEL_ORDER
    proba_matrix = np.column_stack([proba_dict[m] for m in model_order])
    std_feat   = proba_matrix.std(axis=1, keepdims=True)
    mean_feat  = proba_matrix.mean(axis=1, keepdims=True)
    p = np.clip(proba_matrix, 1e-7, 1-1e-7)
    ent_feat   = (-(p * np.log(p) + (1-p) * np.log(1-p))).mean(axis=1, keepdims=True)
    range_feat = (proba_matrix.max(axis=1) - proba_matrix.min(axis=1)).reshape(-1, 1)
    return np.hstack([proba_matrix, std_feat, mean_feat, ent_feat, range_feat])


def _train_and_eval_meta(meta, X_meta_train, y_train, X_meta_test, y_test):
    meta.fit(X_meta_train, y_train)
    train_proba = meta.predict_proba(X_meta_train)[:, 1]
    test_proba  = meta.predict_proba(X_meta_test)[:, 1]
    # A1: meta probability uzerinde threshold optimize et (train OOF prob'larina gore)
    best_thr, _ = optimize_threshold(y_train.values, train_proba)
    test_metrics  = compute_all_metrics(
        y_test.values,  (test_proba  >= best_thr).astype(int), test_proba
    )
    train_metrics = compute_all_metrics(
        y_train.values, (train_proba >= best_thr).astype(int), train_proba
    )
    return meta, test_metrics, test_proba, train_metrics, best_thr


def train_meta_lr(X_meta_tr, y_tr, X_meta_te, y_te):
    meta = LogisticRegression(C=1.0, penalty='l2', class_weight='balanced',
                              max_iter=1000, random_state=SEED)
    return _train_and_eval_meta(meta, X_meta_tr, y_tr, X_meta_te, y_te)


def train_meta_lgbm(X_meta_tr, y_tr, X_meta_te, y_te):
    meta = lgb.LGBMClassifier(n_estimators=100, max_depth=3, learning_rate=0.1,
                              class_weight='balanced', random_state=SEED,
                              verbosity=-1)
    return _train_and_eval_meta(meta, X_meta_tr, y_tr, X_meta_te, y_te)


def stacking_predict(base_results, meta_model, meta_thr, X_panel, model_order=None):
    if model_order is None:
        model_order = BASE_MODEL_ORDER
    panel_probas = {}
    for m in model_order:
        try:
            raw = base_results[m]['predict_fn'](X_panel)
            # apply calibration (B3) -- panel uses same isotonic fit from train
            iso = base_results[m]['iso']
            panel_probas[m] = iso.transform(raw)
        except Exception as e:
            print(f'    Warning: {m} panel predict failed: {e}')
            panel_probas[m] = np.full(len(X_panel), 0.5)
    X_meta_panel = build_meta_features(panel_probas, model_order)
    proba = meta_model.predict_proba(X_meta_panel)[:, 1]
    return proba, panel_probas


print('Meta-learner functions defined. Meta-feature dims = '
      f'{len(BASE_MODEL_ORDER) + 4} ({len(BASE_MODEL_ORDER)} probs + 4 diversity)')


Meta-learner functions defined. Meta-feature dims = 10 (6 probs + 4 diversity)


In [9]:
# Cell 9: Main Experiment Loop
ALL_RESULTS   = []
CONFIG_STATES = {}

checkpoint_path = os.path.join(RESULTS_STACKING_DIR, 'all_results_checkpoint.csv')

SPLIT_FNS = {'B1': make_b1_split, 'B2': make_b2_split}

for bal_id in BALANCE_IDS:
    X_tr_raw, X_te_raw, y_tr, y_te = SPLIT_FNS[bal_id](X_master, y_master)
    print(f'\n{"="*70}')
    print(f'BALANCE: {bal_id}  train={len(y_tr)} '
          f'(pos={int((y_tr==1).sum())}, neg={int((y_tr==0).sum())})  '
          f'test={len(y_te)}')

    for miss_id in MISSING_IDS:
        config_name = f'{bal_id}_{miss_id}'
        print(f'\n{"-"*60}\nCONFIG: {config_name}')
        t0 = time.time()

        prep   = fit_preprocessor(X_tr_raw, miss_id)
        X_tr   = apply_preprocessor(X_tr_raw, prep)
        X_te   = apply_preprocessor(X_te_raw, prep)
        cat_cols = get_surviving_cat_cols(prep)

        # A3: train sonrasi sabit/duplicate sutun temizligi
        redund, consts, dups = detect_redundant_cols(X_tr)
        if redund:
            print(f'  A3 cleanup: drop {len(redund)} cols '
                  f'(consts={len(consts)}, dup_pairs={len(dups)}) examples={redund[:4]}')
            X_tr = X_tr.drop(columns=redund)
            X_te = X_te.drop(columns=[c for c in redund if c in X_te.columns])
            cat_cols = [c for c in cat_cols if c not in redund]

        print(f'  Features: {X_tr.shape[1]} (cat_cols={len(cat_cols)})')

        try:
            base_res = run_all_base_models(X_tr, y_tr, X_te, y_te, cat_cols)
        except Exception as e:
            print(f'  ERROR in base models: {e}')
            import traceback; traceback.print_exc()
            continue

        # B3: kalibre olasiliklari meta'ya ver
        oof_probas  = {m: base_res[m]['oof_proba_calib']  for m in BASE_MODEL_ORDER}
        test_probas = {m: base_res[m]['test_proba_calib'] for m in BASE_MODEL_ORDER}
        X_meta_tr = build_meta_features(oof_probas)
        X_meta_te = build_meta_features(test_probas)

        # A1: meta + threshold opt
        meta_lr,   met_lr,   prob_lr,   met_tr_lr,   thr_lr   = train_meta_lr(
            X_meta_tr, y_tr, X_meta_te, y_te)
        meta_lgbm, met_lgbm, prob_lgbm, met_tr_lgbm, thr_lgbm = train_meta_lgbm(
            X_meta_tr, y_tr, X_meta_te, y_te)

        base_oof_f1s = {}
        for m in BASE_MODEL_ORDER:
            oof_preds = (base_res[m]['oof_proba_calib'] >= FIXED_THR).astype(int)
            base_oof_f1s[m] = f1_score(y_tr, oof_preds, zero_division=0)

        elapsed = time.time() - t0
        print(f'  [LR meta]   Train F1={met_tr_lr["f1"]:.4f}  Test F1={met_lr["f1"]:.4f} '
              f'thr={thr_lr:.2f}  Prec={met_lr["precision"]:.4f}  '
              f'Rec={met_lr["recall"]:.4f}  AUC={met_lr["auc_roc"]:.4f}')
        print(f'  [LGBM meta] Train F1={met_tr_lgbm["f1"]:.4f}  Test F1={met_lgbm["f1"]:.4f} '
              f'thr={thr_lgbm:.2f}  Prec={met_lgbm["precision"]:.4f}  '
              f'Rec={met_lgbm["recall"]:.4f}  AUC={met_lgbm["auc_roc"]:.4f}')
        print(f'  Elapsed: {elapsed/60:.1f} min')

        CONFIG_STATES[config_name] = {
            'prep': prep, 'cat_cols': cat_cols, 'base_res': base_res,
            'meta_lr': meta_lr, 'meta_lgbm': meta_lgbm,
            'thr_lr': thr_lr, 'thr_lgbm': thr_lgbm,
            'train_cols': list(X_tr.columns),
            'redundant_dropped': redund,
        }

        for meta_name, met_test, met_train, thr in [
            ('LR',       met_lr,   met_tr_lr,   thr_lr),
            ('LightGBM', met_lgbm, met_tr_lgbm, thr_lgbm),
        ]:
            row = {
                'config': config_name, 'balance': bal_id, 'missing': miss_id,
                'meta': meta_name, 'n_features': X_tr.shape[1],
                'n_cat_cols': len(cat_cols), 'n_train': len(y_tr), 'n_test': len(y_te),
                'meta_threshold': float(thr),
                'redundant_dropped': len(redund),
            }
            row.update({f'train_{k}': v for k, v in met_train.items()})
            row.update({f'meta_{k}':  v for k, v in met_test.items()})
            for m in BASE_MODEL_ORDER:
                row[f'base_oof_f1_{m}']  = base_oof_f1s[m]
                row[f'base_test_f1_{m}'] = base_res[m]['metrics']['f1']
            ALL_RESULTS.append(row)

        pd.DataFrame(ALL_RESULTS).to_csv(checkpoint_path, index=False)

df_results = pd.DataFrame(ALL_RESULTS)
df_results.to_csv(os.path.join(RESULTS_STACKING_DIR, 'all_results.csv'), index=False)
print(f'\nTotal configs completed: {len(ALL_RESULTS)//2} / {len(BALANCE_IDS) * len(MISSING_IDS)}')
cols_show = ['config', 'meta', 'meta_threshold', 'train_f1', 'meta_f1',
             'meta_precision', 'meta_recall', 'meta_auc_roc']
print(df_results[cols_show].to_string(index=False))



BALANCE: B1  train=2344 (pos=1719, neg=625)  test=587

------------------------------------------------------------
CONFIG: B1_M1
  A3 cleanup: drop 64 cols (consts=57, dup_pairs=1557) examples=['AL_312', 'AL_164', 'AL_208', 'AL_182']
  Features: 287 (cat_cols=7)
[1/6] catboost
    CatBoost best={'depth': 8, 'learning_rate': 0.05} cv_f1=0.8816
  F1=0.8966  AUC=0.8521  (raw_mean=0.714 calib_mean=0.731)
[2/6] lightgbm
    LightGBM best={'num_leaves': 31, 'learning_rate': 0.1, 'n_estimators': 400} cv_f1=0.8665
  F1=0.8945  AUC=0.8321  (raw_mean=0.768 calib_mean=0.716)
[3/6] logreg_l1
    LR-L1 best C=0.1 cv_f1=0.8013
  F1=0.8779  AUC=0.8227  (raw_mean=0.585 calib_mean=0.733)
[4/6] mlp3
  [runner] mlp3 grid: 2 combos
  [runner] combo 0 fold 0: vf=0.8327 elapsed=1.1s
  [runner] combo 0 fold 1: vf=0.8560 elapsed=1.4s
  [runner] combo 1 fold 0: vf=0.7935 elapsed=1.7s
  [runner] combo 1 fold 1: vf=0.8405 elapsed=2.0s
  [runner] best={'dropout': 0.2, 'lr': 0.001} cv_f1=0.8443
  [runner] oof fo

In [10]:
# Cell 10: Panel Transfer Evaluation
PANEL_DATA = {
    'CFTR':   (X_cftr,   y_cftr),
    'KANSER': (X_kanser, y_kanser),
    'PAH':    (X_pah,    y_pah),
}

panel_rows = []
for config_name, state in CONFIG_STATES.items():
    bal_id, miss_id = config_name.split('_')
    prep      = state['prep']
    base_res  = state['base_res']
    meta_lr   = state['meta_lr']
    meta_lgbm = state['meta_lgbm']
    thr_lr    = state['thr_lr']
    thr_lgbm  = state['thr_lgbm']
    train_cols = state['train_cols']
    redund     = state['redundant_dropped']

    for panel_name, (X_panel_raw, y_panel) in PANEL_DATA.items():
        try:
            X_panel = apply_preprocessor(X_panel_raw, prep)
            X_panel = X_panel.drop(columns=[c for c in redund if c in X_panel.columns])

            for c in train_cols:
                if c not in X_panel.columns:
                    X_panel[c] = 0
            X_panel = X_panel[train_cols]

            for meta_name, meta_model, meta_thr in [
                ('LR', meta_lr, thr_lr),
                ('LightGBM', meta_lgbm, thr_lgbm),
            ]:
                proba, _ = stacking_predict(base_res, meta_model, meta_thr, X_panel)
                y_pred = (proba >= meta_thr).astype(int)
                if len(np.unique(y_panel)) < 2:
                    continue
                met = compute_all_metrics(y_panel.values, y_pred, proba)
                cm  = confusion_matrix(y_panel.values, y_pred)
                row = {
                    'config': config_name, 'balance': bal_id, 'missing': miss_id,
                    'meta': meta_name, 'panel': panel_name,
                    'meta_threshold': float(meta_thr),
                    'tn': int(cm[0,0]), 'fp': int(cm[0,1]),
                    'fn': int(cm[1,0]), 'tp': int(cm[1,1]),
                }
                row.update({f'panel_{k}': v for k, v in met.items()})
                panel_rows.append(row)
                print(f'  {config_name}/{panel_name}/{meta_name}: '
                      f'F1={met["f1"]:.4f}  Rec={met["recall"]:.4f}')
        except Exception as e:
            print(f'  ERROR {config_name}/{panel_name}: {e}')
            import traceback; traceback.print_exc()

df_panels = pd.DataFrame(panel_rows)
df_panels.to_csv(os.path.join(RESULTS_STACKING_DIR, 'panel_results.csv'), index=False)
print(f'\nPanel results saved. {len(df_panels)} rows.')


  B1_M1/CFTR/LR: F1=0.9451  Rec=0.9556
  B1_M1/CFTR/LightGBM: F1=0.9392  Rec=0.9444
  B1_M1/KANSER/LR: F1=0.8983  Rec=0.9888
  B1_M1/KANSER/LightGBM: F1=0.8998  Rec=0.9888
  B1_M1/PAH/LR: F1=0.9329  Rec=0.9871
  B1_M1/PAH/LightGBM: F1=0.9362  Rec=0.9935
  B1_M3/CFTR/LR: F1=0.9438  Rec=0.9333
  B1_M3/CFTR/LightGBM: F1=0.9379  Rec=0.9222
  B1_M3/KANSER/LR: F1=0.9085  Rec=0.9813
  B1_M3/KANSER/LightGBM: F1=0.9094  Rec=0.9739
  B1_M3/PAH/LR: F1=0.9290  Rec=0.9710
  B1_M3/PAH/LightGBM: F1=0.9279  Rec=0.9548
  B1_M5/CFTR/LR: F1=0.9508  Rec=0.9667
  B1_M5/CFTR/LightGBM: F1=0.9266  Rec=0.9111
  B1_M5/KANSER/LR: F1=0.9010  Rec=0.9851
  B1_M5/KANSER/LightGBM: F1=0.9101  Rec=0.9627
  B1_M5/PAH/LR: F1=0.9313  Rec=0.9839
  B1_M5/PAH/LightGBM: F1=0.9079  Rec=0.9226
  B2_M1/CFTR/LR: F1=0.8876  Rec=0.8333
  B2_M1/CFTR/LightGBM: F1=0.8941  Rec=0.8444
  B2_M1/KANSER/LR: F1=0.9097  Rec=0.9403
  B2_M1/KANSER/LightGBM: F1=0.9078  Rec=0.9366
  B2_M1/PAH/LR: F1=0.9250  Rec=0.9548
  B2_M1/PAH/LightGBM: F1=0.9

In [11]:
# Cell 11: Results Compilation
df_lr        = df_results[df_results['meta'] == 'LR'].sort_values('meta_f1', ascending=False)
df_lgbm_meta = df_results[df_results['meta'] == 'LightGBM'].sort_values('meta_f1', ascending=False)

cols_show = ['config', 'meta_threshold', 'meta_f1', 'meta_precision',
             'meta_recall', 'meta_auc_roc', 'meta_mcc']
print('=== TOP 5 CONFIGS (LR Meta) ===')
print(df_lr[cols_show].head(5).to_string(index=False))
print('\n=== TOP 5 CONFIGS (LightGBM Meta) ===')
print(df_lgbm_meta[cols_show].head(5).to_string(index=False))

best_config = df_lr.iloc[0]['config'] if len(df_lr) > 0 else 'B1_M1'
print(f'\nBest config overall (LR): {best_config}')

print('\n=== MISSING SCENARIO COMPARISON (avg F1 across balance) ===')
if len(df_results) > 0:
    miss_avg = df_results.groupby(['missing', 'meta'])['meta_f1'].mean().unstack()
    print(miss_avg.round(4))

print('\n=== BALANCE COMPARISON ===')
if len(df_results) > 0:
    bal_avg = df_results.groupby(['balance', 'meta'])['meta_f1'].mean().unstack()
    print(bal_avg.round(4))

print('\n=== A2 NB12 baseline check ===')
print(f'  NB14-reproduced XGB+OHE F1={NB12_BASELINE["f1"]:.4f} '
      f'AUC={NB12_BASELINE["auc"]:.4f}  thr={NB12_BASELINE["thr"]:.3f}')
print(f'  NB12 reference: F1=0.9020  AUC=0.8404')


=== TOP 5 CONFIGS (LR Meta) ===
config  meta_threshold  meta_f1  meta_precision  meta_recall  meta_auc_roc  meta_mcc
 B1_M1            0.19 0.894220        0.841889     0.953488      0.863620  0.545181
 B1_M5            0.17 0.893757        0.844720     0.948837      0.866812  0.546179
 B1_M3            0.20 0.893054        0.849057     0.941860      0.863583  0.548165
 B2_M3            0.35 0.880274        0.863535     0.897674      0.856051  0.528814
 B2_M1            0.38 0.873832        0.877934     0.869767      0.852459  0.534316

=== TOP 5 CONFIGS (LightGBM Meta) ===
config  meta_threshold  meta_f1  meta_precision  meta_recall  meta_auc_roc  meta_mcc
 B1_M1            0.25 0.893524        0.846154     0.946512      0.848415  0.546778
 B1_M3            0.28 0.890625        0.856223     0.927907      0.850415  0.548366
 B2_M3            0.39 0.883774        0.874715     0.893023      0.851074  0.553203
 B2_M5            0.33 0.879271        0.861607     0.897674      0.854348  0.5

In [12]:
# Cell 12: Visualizations
FIG_DIR = RESULTS_STACKING_DIR

# Figure 1: MASTER F1/Precision/Recall per config x meta
if len(df_results) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    configs = df_results['config'].unique()
    x = np.arange(len(configs))
    width = 0.35
    for ax, metric in zip(axes, ['meta_f1', 'meta_precision', 'meta_recall']):
        for i, meta_name in enumerate(['LR', 'LightGBM']):
            df_sub = df_results[df_results['meta'] == meta_name].set_index('config')
            vals = [df_sub.loc[c, metric] if c in df_sub.index else 0 for c in configs]
            ax.bar(x + i*width, vals, width, label=meta_name, alpha=0.8)
        ax.set_xticks(x + width/2)
        ax.set_xticklabels(configs, rotation=45, ha='right', fontsize=9)
        ax.set_ylabel(metric.replace('meta_', '').upper())
        ax.set_title(f'{metric.replace("meta_", "").title()} by Config x Meta')
        ax.legend(); ax.set_ylim(0, 1.05); ax.grid(axis='y', alpha=0.3)
    plt.suptitle('MASTER Hold-Out Results (calibrated probs, threshold-tuned meta)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    fig1_path = os.path.join(FIG_DIR, 'fig1_master_metrics.png')
    plt.savefig(fig1_path, dpi=120, bbox_inches='tight'); plt.close()
    print(f'Figure 1 saved: {fig1_path}')

# Figure 2: Base model F1 (best config)
if best_config in CONFIG_STATES:
    bc_res = CONFIG_STATES[best_config]['base_res']
    test_f1s = [bc_res[m]['metrics']['f1'] for m in BASE_MODEL_ORDER]
    stack_f1 = df_results[(df_results['config']==best_config) &
                          (df_results['meta']=='LR')]['meta_f1'].values
    stack_f1_val = float(stack_f1[0]) if len(stack_f1) > 0 else 0.0
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(np.arange(len(BASE_MODEL_ORDER)), test_f1s,
                   color='steelblue', alpha=0.8)
    ax.axhline(stack_f1_val, color='red', linestyle='--', linewidth=2,
               label=f'Stacking LR F1={stack_f1_val:.4f}')
    ax.set_xticks(np.arange(len(BASE_MODEL_ORDER)))
    ax.set_xticklabels([m.upper() for m in BASE_MODEL_ORDER], rotation=20)
    ax.set_ylabel('F1 Score'); ax.set_ylim(0, 1.05)
    ax.set_title(f'Base Model Test F1 vs Stacking ({best_config})')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    for bar, f1 in zip(bars, test_f1s):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{f1:.3f}', ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    fig2_path = os.path.join(FIG_DIR, 'fig2_base_model_f1.png')
    plt.savefig(fig2_path, dpi=120, bbox_inches='tight'); plt.close()
    print(f'Figure 2 saved: {fig2_path}')

# Figure 3: Calibration effect (raw vs calibrated mean prob, best config)
if best_config in CONFIG_STATES:
    bc_res = CONFIG_STATES[best_config]['base_res']
    raw_means   = [bc_res[m]['test_proba_raw'].mean()   for m in BASE_MODEL_ORDER]
    calib_means = [bc_res[m]['test_proba_calib'].mean() for m in BASE_MODEL_ORDER]
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(BASE_MODEL_ORDER)); w = 0.35
    ax.bar(x - w/2, raw_means,   w, label='Raw',        color='gray',     alpha=0.7)
    ax.bar(x + w/2, calib_means, w, label='Calibrated', color='darkgreen',alpha=0.7)
    ax.set_xticks(x); ax.set_xticklabels([m.upper() for m in BASE_MODEL_ORDER], rotation=20)
    ax.set_ylabel('Mean probability on test set')
    ax.set_title(f'B3 Calibration Effect (best config: {best_config})')
    ax.legend(); ax.grid(axis='y', alpha=0.3); ax.set_ylim(0, 1.0)
    plt.tight_layout()
    fig3_path = os.path.join(FIG_DIR, 'fig3_calibration_effect.png')
    plt.savefig(fig3_path, dpi=120, bbox_inches='tight'); plt.close()
    print(f'Figure 3 saved: {fig3_path}')

# Figure 4: Confusion matrices (best config, LR meta)
if best_config in CONFIG_STATES:
    state_bc = CONFIG_STATES[best_config]
    meta_lr_bc = state_bc['meta_lr']
    thr_bc = state_bc['thr_lr']
    bc_bal = best_config.split('_')[0]
    _, X_te_bc_raw, _, y_te_bc = SPLIT_FNS[bc_bal](X_master, y_master)
    X_te_bc = apply_preprocessor(X_te_bc_raw, state_bc['prep'])
    redund_bc = state_bc['redundant_dropped']
    X_te_bc = X_te_bc.drop(columns=[c for c in redund_bc if c in X_te_bc.columns])
    for c in state_bc['train_cols']:
        if c not in X_te_bc.columns:
            X_te_bc[c] = 0
    X_te_bc = X_te_bc[state_bc['train_cols']]
    test_p_bc, _ = stacking_predict(state_bc['base_res'], meta_lr_bc, thr_bc, X_te_bc)
    y_pred_bc = (test_p_bc >= thr_bc).astype(int)
    cm_master = confusion_matrix(y_te_bc.values, y_pred_bc)
    panels_for_cm = [(p, df_panels[(df_panels['config']==best_config) &
                                    (df_panels['meta']=='LR') &
                                    (df_panels['panel']==p)])
                     for p in ['CFTR', 'KANSER', 'PAH']]
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    ConfusionMatrixDisplay(cm_master, display_labels=['Benign','Pathogenic']).plot(
        ax=axes[0], colorbar=False)
    axes[0].set_title(f'MASTER Test\n(config={best_config}, thr={thr_bc:.2f})',
                       fontsize=10)
    for ax, (pname, pdf) in zip(axes[1:], panels_for_cm):
        if len(pdf) > 0:
            row = pdf.iloc[0]
            cm_p = np.array([[int(row['tn']), int(row['fp'])],
                             [int(row['fn']), int(row['tp'])]])
            ConfusionMatrixDisplay(cm_p, display_labels=['Benign','Pathogenic']).plot(
                ax=ax, colorbar=False)
        ax.set_title(f'{pname} Panel', fontsize=10)
    plt.suptitle(f'Confusion Matrices - {best_config} (LR Meta, thr={thr_bc:.2f})',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    fig4_path = os.path.join(FIG_DIR, 'fig4_confusion_matrices.png')
    plt.savefig(fig4_path, dpi=120, bbox_inches='tight'); plt.close()
    print(f'Figure 4 saved: {fig4_path}')

print('Figures generated.')


Figure 1 saved: /Users/tefe/teknofest_model/teknofest_model/results/v4_stacking/fig1_master_metrics.png
Figure 2 saved: /Users/tefe/teknofest_model/teknofest_model/results/v4_stacking/fig2_base_model_f1.png
Figure 3 saved: /Users/tefe/teknofest_model/teknofest_model/results/v4_stacking/fig3_calibration_effect.png
Figure 4 saved: /Users/tefe/teknofest_model/teknofest_model/results/v4_stacking/fig4_confusion_matrices.png
Figures generated.


In [13]:
# Cell 13: PDF Report
class StackingReport(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 10)
        self.cell(0, 8, 'NB14 v2 - Stacking Ensemble | TEKNOFEST Genetik Varyant',
                  ln=True, align='C')
        self.ln(2)

    def footer(self):
        self.set_y(-12)
        self.set_font('Helvetica', 'I', 8)
        self.cell(0, 8, f'Sayfa {self.page_no()}', align='C')

    def chapter_title(self, title):
        self.set_font('Helvetica', 'B', 12)
        self.set_fill_color(50, 100, 200); self.set_text_color(255, 255, 255)
        self.cell(0, 8, title, ln=True, fill=True)
        self.set_text_color(0, 0, 0); self.ln(3)

    def section_title(self, title):
        self.set_font('Helvetica', 'B', 10)
        self.set_fill_color(200, 220, 255)
        self.cell(0, 7, title, ln=True, fill=True); self.ln(2)

    def body_text(self, text):
        self.set_font('Helvetica', '', 9)
        self.multi_cell(0, 5, text); self.ln(2)

    def table_header(self, cols, widths):
        self.set_font('Helvetica', 'B', 8)
        self.set_fill_color(70, 130, 180); self.set_text_color(255, 255, 255)
        for col, w in zip(cols, widths):
            self.cell(w, 6, str(col), border=1, fill=True)
        self.ln(); self.set_text_color(0, 0, 0)

    def table_row(self, vals, widths, fill=False):
        self.set_font('Helvetica', '', 7)
        if fill:
            self.set_fill_color(235, 243, 255)
        for val, w in zip(vals, widths):
            self.cell(w, 5, str(val), border=1, fill=fill)
        self.ln()


def build_report():
    pdf = StackingReport(orientation='L', format='A4')
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()

    pdf.set_font('Helvetica', 'B', 18); pdf.ln(10)
    pdf.cell(0, 12, 'NB14 v2: Stacking Ensemble (5-Model Diversity)', ln=True, align='C')
    pdf.set_font('Helvetica', '', 12)
    pdf.cell(0, 8, 'TEKNOFEST Saglikta Yapay Zeka | Genetik Varyant',
             ln=True, align='C')
    pdf.cell(0, 8, f'Tarih: {datetime.now().strftime("%Y-%m-%d %H:%M")}',
             ln=True, align='C')
    pdf.ln(6)

    pdf.section_title('Degisiklik Ozeti (v2)')
    pdf.body_text(
        'A1) Meta probability uzerinde threshold optimizasyonu (F1-max).\n'
        'A2) NB12 XGBoost+OHE baseline yeniden uretildi: '
        f'F1={NB12_BASELINE["f1"]:.4f}, AUC={NB12_BASELINE["auc"]:.4f}, '
        f'thr={NB12_BASELINE["thr"]:.3f}. NB12 referans: F1=0.9020 AUC=0.8404.\n'
        'A3) Sabit sutunlar ve birebir ayni sutun ciftleri (CAT_3/CAT_5 dahil) '
        'train sonrasi drop edildi.\n'
        'B1) Taban modeller 7 -> 5 cesitliligi: CatBoost (native cat), LightGBM '
        '(LE), Logistic L1, TabNet, kNN.\n'
        'B3) OOF probability uzerinde IsotonicRegression; meta-feature matrisi '
        'kalibre olasiliklardan kuruluyor.\n'
        'Balance B2 duzeltildi: B1 ile ayni test seti, train icinde patho. '
        'undersample, fazla patho. DROP (test"e tasinmaz, ID overlap=0).'
    )

    pdf.add_page()
    pdf.chapter_title('1. Sonuclar (MASTER Hold-Out)')
    if len(df_results) > 0:
        cols_r  = ['Config', 'Meta', 'Thr',
                   'Tr-F1', 'Te-F1', 'Te-Prec', 'Te-Rec', 'Te-AUC', 'Te-MCC', 'N-feat']
        widths_r = [22, 20, 14, 22, 22, 22, 22, 22, 22, 18]
        pdf.table_header(cols_r, widths_r)
        for i, row in df_results.sort_values(['config', 'meta']).iterrows():
            vals_r = [
                row['config'], row['meta'], f"{row['meta_threshold']:.2f}",
                f"{row.get('train_f1', float('nan')):.4f}",
                f"{row['meta_f1']:.4f}", f"{row['meta_precision']:.4f}",
                f"{row['meta_recall']:.4f}", f"{row['meta_auc_roc']:.4f}",
                f"{row['meta_mcc']:.4f}", str(int(row['n_features'])),
            ]
            pdf.table_row(vals_r, widths_r, fill=(i % 2 == 0))

    pdf.add_page()
    pdf.chapter_title(f'2. En Iyi Konfigurasyon: {best_config}')
    if best_config in CONFIG_STATES:
        bc_res = CONFIG_STATES[best_config]['base_res']
        cols_b  = ['Model', 'Test F1 (calib)', 'Test AUC', 'OOF F1 (calib)']
        widths_b = [50, 50, 50, 50]
        pdf.table_header(cols_b, widths_b)
        bc_ytr_pdf = SPLIT_FNS[best_config.split('_')[0]](X_master, y_master)[2]
        for i, m in enumerate(BASE_MODEL_ORDER):
            f1_oof_pdf = f1_score(bc_ytr_pdf,
                                  (bc_res[m]['oof_proba_calib'] >= 0.5).astype(int),
                                  zero_division=0)
            pdf.table_row(
                [m, f"{bc_res[m]['metrics']['f1']:.4f}",
                 f"{bc_res[m]['metrics']['auc_roc']:.4f}",
                 f"{f1_oof_pdf:.4f}"],
                widths_b, fill=(i % 2 == 0)
            )
        best_stack_lr = df_results[(df_results['config'] == best_config) &
                                    (df_results['meta'] == 'LR')]
        if len(best_stack_lr) > 0:
            r0 = best_stack_lr.iloc[0]
            pdf.ln(4)
            pdf.body_text(
                f'Stacking LR (calibrated + threshold-tuned) test F1: '
                f'{r0["meta_f1"]:.4f}  thr={r0["meta_threshold"]:.2f}\n'
                f'A3 redundant drops at best config: '
                f'{CONFIG_STATES[best_config]["redundant_dropped"]}'
            )

    if len(df_panels) > 0:
        pdf.add_page()
        pdf.chapter_title('3. Panel Transfer Sonuclari (Top-5 by MASTER F1)')
        top5 = df_results[df_results['meta'] == 'LR'].sort_values(
            'meta_f1', ascending=False).head(5)['config'].tolist()
        df_p_top = df_panels[(df_panels['config'].isin(top5)) &
                              (df_panels['meta'] == 'LR')]
        cols_p  = ['Config', 'Panel', 'Thr', 'F1', 'Recall', 'Precision', 'AUC', 'MCC']
        widths_p = [28, 22, 16, 26, 26, 28, 26, 24]
        pdf.table_header(cols_p, widths_p)
        for i, row in df_p_top.sort_values(['config', 'panel']).iterrows():
            pdf.table_row(
                [row['config'], row['panel'], f"{row['meta_threshold']:.2f}",
                 f"{row['panel_f1']:.4f}", f"{row['panel_recall']:.4f}",
                 f"{row['panel_precision']:.4f}", f"{row['panel_auc_roc']:.4f}",
                 f"{row['panel_mcc']:.4f}"],
                widths_p, fill=(i % 2 == 0)
            )

    for fname, title in [
        ('fig1_master_metrics.png',     '4. MASTER F1 / Precision / Recall'),
        ('fig2_base_model_f1.png',      '5. Taban Modeller vs Stacking'),
        ('fig3_calibration_effect.png', '6. B3 Calibration Etkisi'),
        ('fig4_confusion_matrices.png', '7. Konfuzyon Matrisleri'),
    ]:
        path_p = os.path.join(RESULTS_STACKING_DIR, fname)
        if os.path.exists(path_p):
            pdf.add_page()
            pdf.chapter_title(title)
            pdf.image(path_p, x=10, y=40, w=270)

    pdf_path = os.path.join(REPORTS_DIR, 'NB14_stacking_report.pdf')
    pdf.output(pdf_path)
    print(f'PDF raporu kaydedildi: {pdf_path}')
    return pdf_path


pdf_path = build_report()
print(f'Rapor tamamlandi: {pdf_path}')


PDF raporu kaydedildi: /Users/tefe/teknofest_model/teknofest_model/reports/NB14_stacking_report.pdf
Rapor tamamlandi: /Users/tefe/teknofest_model/teknofest_model/reports/NB14_stacking_report.pdf
